In [0]:
from pyspark.sql import functions as F

ID_COLS = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]

def build_silver(spark):
    """Wide daily sales -> one row per id x date, with calendar, price and revenue.
    Drops the days before each product's first sale (M5 fills those with 0)."""
    wide = spark.table("workspace.m5.bronze_sales_wide")
    day_cols = [c for c in wide.columns if c.startswith("d_")]

    # 1) wide -> long
    long = wide.unpivot(ids=ID_COLS, values=day_cols,
                        variableColumnName="d", valueColumnName="units")

    # 2) add date/week from calendar, price from prices
    cal = (spark.table("workspace.m5.bronze_calendar")
                .select("d", F.to_date("date").alias("date"), "wm_yr_wk", "weekday", "event_name_1"))
    prices = spark.table("workspace.m5.bronze_prices")

    daily = (long.join(cal, "d")
                 .join(prices, ["store_id", "item_id", "wm_yr_wk"], "left")
                 .withColumn("revenue", F.col("units") * F.col("sell_price")))

    # 3) remove pre-launch days: before the first sale the product wasn't on sale yet
    first_sale = daily.filter("units > 0").groupBy("id").agg(F.min("date").alias("first_sale_date"))
    return (daily.join(first_sale, "id")
                 .filter(F.col("date") >= F.col("first_sale_date"))
                 .drop("d", "first_sale_date"))